# EXXA Test Submission — End-to-End Pipeline

This notebook runs the full PLANAR pipeline from start to finish (General + Image + Sequential tests), produces all required plots/metrics, and demonstrates withheld-data inference.


In [ ]:
# Ensure local package imports work
import os
import sys
from pathlib import Path
root = Path.cwd()
while root != root.parent and not (root / "src" / "planar").exists():
    root = root.parent
os.chdir(root)
src = root / "src"
for entry in ("", str(root)):
    while entry in sys.path:
        sys.path.remove(entry)
if str(src) not in sys.path:
    sys.path.insert(0, str(src))
print("Using repo root:", root)


In [ ]:
# Clone repo and install dependencies (Colab-friendly)
import os
from pathlib import Path
in_colab = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
repo_root = Path.cwd()
if not (repo_root / "src" / "planar").exists():
    !git clone https://github.com/Atharva12081/PLANAR.git
    %cd PLANAR
    repo_root = Path.cwd()
if in_colab:
    !pip install -r requirements.txt
    !pip install -e .


In [ ]:
# Verify MS-SSIM dependency is available
from planar.metrics import HAS_MS_SSIM
print('MS-SSIM available:', HAS_MS_SSIM)


## Run Full Pipeline
This runs: autoencoder training → clustering → transit classifier → inference → report.


In [ ]:
!python -m planar run --config configs/verify.yaml


## Inspect Outputs & Metrics


In [ ]:
from pathlib import Path
import json
art = Path("artifacts")
print("Autoencoder summary:")
print(json.loads((art/"autoencoder"/"train_summary.json").read_text()))
print("\nClustering summary:")
print(json.loads((art/"clustering"/"clustering_summary.json").read_text()))
print("\nTransit summary:")
print(json.loads((art/"transit"/"train_summary.json").read_text()))


In [ ]:
# Visualizations
from IPython.display import Image, display

for p in [
    'artifacts/autoencoder/recon_examples.png',
    'artifacts/autoencoder/loss_curve.png',
    'artifacts/clustering/latent_scatter.png',
    'artifacts/clustering/cluster_means.png',
    'artifacts/transit/roc_curve.png',
    'artifacts/transit/stress_roc_curve.png',
]:
    display(Image(filename=p))


## Withheld Data Test (New FITS Files)
Generate new synthetic FITS files and run inference on them.


In [ ]:
# Generate new synthetic FITS
!python scripts/generate_synthetic_fits.py --out-dir data/withheld_fits --n-samples 8

# Run inference on withheld data
!python -m planar infer --config configs/verify.yaml --data-dir data/withheld_fits


## Withheld Data Test (Observational Light Curves)
Create a small labeled observational-style dataset (CSV) and evaluate the classifier on it.


In [ ]:
# Run full pipeline with observational eval enabled
!python -m planar run --config configs/verify_observational.yaml


## End
This notebook completes the required EXXA pipeline and demonstrates withheld-data processing.
